# ADVERTISING SALES PREDICTION & IMPACT ANALYSIS
Project: Predictive Analytics for Marketing Strategy Optimization

PROBLEM STATEMENT:
==================
Organizations struggle with:
1. Uncertainty about ROI for each advertising channel
2. Inefficient budget allocation across TV, Radio, and Newspaper advertising
3. Inability to forecast sales based on planned advertising expenditure
4. Limited understanding of channel synergies and individual channel impact
5. Lack of data-driven strategies for marketing optimization
 
OBJECTIVES:
===========
1. Build predictive models to forecast sales based on advertising spend
2. Identify and quantify the impact of each advertising channel on sales
3. Perform comprehensive data cleaning, transformation, and feature engineering
4. Conduct elasticity analysis to understand channel sensitivity
5. Generate actionable business recommendations for marketing strategy

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split,cross_val_score,KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
import warnings
warnings.filterwarnings('ignore')

- Importing Libraries

# DATA LOADING & EXPLORATION 

In [2]:
df= pd.read_csv('Advertising.csv')
df.head()

,Unnamed: 0,TV,Radio,Newspaper,Sales
0,1,230.1,37.8,69.2,22.1
1,2,44.5,39.3,45.1,10.4
2,3,17.2,45.9,69.3,9.3
3,4,151.5,41.3,58.5,18.5
4,5,180.8,10.8,58.4,12.9


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  200 non-null    int64  
 1   TV          200 non-null    float64
 2   Radio       200 non-null    float64
 3   Newspaper   200 non-null    float64
 4   Sales       200 non-null    float64
dtypes: float64(4), int64(1)
memory usage: 7.9 KB


In [4]:
df.dtypes

Unnamed: 0      int64
TV            float64
Radio         float64
Newspaper     float64
Sales         float64
dtype: object

In [5]:
df.describe()

,Unnamed: 0,TV,Radio,Newspaper,Sales
count,200.000000,200.000000,200.000000,200.000000,200.000000
mean,100.500000,147.042500,23.264000,30.554000,14.022500
std,57.879185,85.854236,14.846809,21.778621,5.217457
min,1.000000,0.700000,0.000000,0.300000,1.600000
25%,50.750000,74.375000,9.975000,12.750000,10.375000
50%,100.500000,149.750000,22.900000,25.750000,12.900000
75%,150.250000,218.825000,36.525000,45.100000,17.400000
max,200.000000,296.400000,49.600000,114.000000,27.000000


- Display dataset information

In [6]:
df.isnull().sum()

Unnamed: 0    0
TV            0
Radio         0
Newspaper     0
Sales         0
dtype: int64

- Check For Missing Values

# EXPLORATORY DATA ANALYSIS

In [7]:
# Calculate additional statistics
print("\nSTATISTICAL SUMMARY:")
correlation_matrix=df.corr()
print(correlation_matrix)
# Identify strongest correlations with Sales
print(f"\nChannel Impact on Sales (Correlation with Sales):")
channel_impacts=df.corr()['Sales'].sort_values(ascending=False)
print(channel_impacts)


STATISTICAL SUMMARY:
            Unnamed: 0        TV     Radio  Newspaper     Sales
Unnamed: 0    1.000000  0.017715 -0.110680  -0.154944 -0.051616
TV            0.017715  1.000000  0.054809   0.056648  0.782224
Radio        -0.110680  0.054809  1.000000   0.354104  0.576223
Newspaper    -0.154944  0.056648  0.354104   1.000000  0.228299
Sales        -0.051616  0.782224  0.576223   0.228299  1.000000

Channel Impact on Sales (Correlation with Sales):
Sales         1.000000
TV            0.782224
Radio         0.576223
Newspaper     0.228299
Unnamed: 0   -0.051616
Name: Sales, dtype: float64


# DATA CLEANING & TRANSFORMATION

In [9]:
# Create a copy for transformation
df_processed=df.copy()

In [11]:
# Remove outliers using IQR method
# Remove outliers using Interquartile Range (IQR) method.
#     IQR method: Outliers are values beyond Q3 + 1.5*IQR or below Q1 - 1.5*IQR
#     This is a robust statistical approach for outlier detection.
def remove_outliers(dataframe,columns):
    outlier_indices=[]
    for col in columns:
        Q1= dataframe[col].quantile(0.25)
        Q3= dataframe[col].quantile(0.75)
        IQR=Q3-Q1
        lower_bound= Q1- 1.5*IQR
        upper_bound= Q3+ 1.5*IQR
        outliers=dataframe[(dataframe[col]<lower_bound) | (dataframe[col] > upper_bound)].index
        outlier_indices.extend(outliers)
    return list(set(outlier_indices))
outlier_indices= remove_outliers(df_processed,['TV','Radio','Newspaper','Sales'])
print(f"Outliers detected: {len(outlier_indices)} records")
print(f"Outlier indices: {sorted(outlier_indices)}")

Outliers detected: 2 records
Outlier indices: [16, 101]


# FEATURE ENGINEERING

In [14]:
# Create interaction features
df['TV_radio_interaction']=df_processed['TV']*df_processed['Radio']
df_processed['TV_Newspaper_Interaction'] = df_processed['TV'] * df_processed['Newspaper']
df_processed['Radio_Newspaper_Interaction'] = df_processed['Radio'] * df_processed['Newspaper']

# Create total advertising spend feature
df_processed['Total_Ad_Spend']= df_processed['TV'] + df_processed['Radio'] + df_processed['Newspaper']

# Create channel proportion features (channel mix analysis)
df_processed['TV_Proportion'] = df_processed['TV'] / df_processed['Total_Ad_Spend']
df_processed['Radio_Proportion'] = df_processed['Radio'] / df_processed['Total_Ad_Spend']
df_processed['Newspaper_Proportion'] = df_processed['Newspaper'] / df_processed['Total_Ad_Spend']

print("✓ Created interaction features:")
print("  - TV_Radio_Interaction (TV × Radio)")
print("  - TV_Newspaper_Interaction (TV × Newspaper)")
print("  - Radio_Newspaper_Interaction (Radio × Newspaper)")
print("\n✓ Created aggregate features:")
print("  - Total_Ad_Spend (sum of all channels)")
print("\n✓ Created proportion features:")
print("  - TV_Proportion, Radio_Proportion, Newspaper_Proportion")

✓ Created interaction features:
  - TV_Radio_Interaction (TV × Radio)
  - TV_Newspaper_Interaction (TV × Newspaper)
  - Radio_Newspaper_Interaction (Radio × Newspaper)

✓ Created aggregate features:
  - Total_Ad_Spend (sum of all channels)

✓ Created proportion features:
  - TV_Proportion, Radio_Proportion, Newspaper_Proportion


In [19]:
# Feature selection
# Prepare data for modeling
X= df_processed.drop('Sales',axis=1)
y= df_processed['Sales']

# Calculate correlation of all features with target
feature_correlation= X.corrwith(y).sort_values(ascending=False)
print(f"\nFeature Correlation with Sales:")
for feature,corr in feature_correlation.items():
    strength= "Very Strong" if abs(corr) > 0.8 else "Strong" if abs(corr)>0.6 else "Moderate"if abs(corr) > 0.4 else "Weak"
    print(f"{feature:30} : {corr:7.4f} ({strength})")

# Select top features (correlation > 0.4)
selected_features = feature_correlation[abs(feature_correlation)>0.4].index.tolist()
print(f"\n✓ Selected {len(selected_features)} features with |correlation| > 0.4")
print(f"Selected features: {selected_features}")


Feature Correlation with Sales:
Total_Ad_spend                 :  0.8677 (Very Strong)
Total_Ad_Spend                 :  0.8677 (Very Strong)
TV                             :  0.7822 (Strong)
TV_Newspaper_Interaction       :  0.6185 (Strong)
Radio                          :  0.5762 (Moderate)
TV_Proportion                  :  0.4472 (Moderate)
Radio_Newspaper_Interaction    :  0.4159 (Moderate)
Newspaper                      :  0.2283 (Weak)
Unnamed: 0                     : -0.0516 (Weak)
Radio_Proportion               : -0.2675 (Weak)
Newspaper_Proportion           : -0.4400 (Moderate)

✓ Selected 8 features with |correlation| > 0.4
Selected features: ['Total_Ad_spend', 'Total_Ad_Spend', 'TV', 'TV_Newspaper_Interaction', 'Radio', 'TV_Proportion', 'Radio_Newspaper_Interaction', 'Newspaper_Proportion']


# DATA STANDARDIZATION

In [22]:
# Standardize features for better model performance
scaler= StandardScaler()
X_scaled= scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

Standardization (z-score normalization) transforms features to have:
  - Mean = 0
  - Standard Deviation = 1
 
Benefits:
  1. Improves model convergence for distance-based algorithms
  2. Prevents features with large scales from dominating the model
  3. Essential for Ridge/Lasso regression (L1/L2 regularization)
  4. Enables fair feature importance comparison
  
Formula: Z = (X - mean) / std_dev

# TRAIN-TEST SPLIT

In [29]:
# Use original features for main models
X_main = df[['TV', 'Radio', 'Newspaper']]
y_main = df['Sales']

# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X_main, y_main, test_size=0.2, random_state=42
)

# Also create scaled versions
X_train_scaled, X_test_scaled, _, _ = train_test_split(
    X_scaled_df, y_main, test_size=0.2, random_state=42
)

#  MODEL BUILDING & TRAINING

In [30]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression (α=1.0)': Ridge(alpha=1.0),
    'Lasso Regression (α=0.1)': Lasso(alpha=0.1),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42, learning_rate=0.1)
}

MODEL EXPLANATIONS:
 
1. LINEAR REGRESSION:
   - Assumes linear relationship: Sales = β₀ + β₁(TV) + β₂(Radio) + β₃(Newspaper)
   - Interpretable: coefficients show direct impact of each channel
   - Best for: Understanding channel elasticity
 
2. RIDGE REGRESSION:
   - Linear model with L2 regularization (penalty on large coefficients)
   - Reduces overfitting by constraining coefficient magnitudes
   - Better generalization on test data
 
3. LASSO REGRESSION:
   - Linear model with L1 regularization
   - Can shrink some coefficients to zero (feature selection)
   - Identifies most important channels
 
4. RANDOM FOREST:
   - Ensemble of decision trees
   - Captures non-linear relationships and interactions automatically
   - Provides feature importance rankings
   - Better for: Complex non-linear patterns
 
5. GRADIENT BOOSTING:
   - Sequential ensemble learning
   - Each tree corrects previous trees' errors
   - Often provides best predictive performance
   - Slower but typically most accurate

In [31]:
# Train models
trained_models = {}
print("\n7.2 MODEL TRAINING RESULTS:")
print("-" * 80)
 
for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    # Make predictions
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Calculate metrics
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    train_mae = mean_absolute_error(y_train, y_pred_train)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    
    print(f"\n{name}:")
    print(f"  Training R²: {train_r2:.4f} | Testing R²: {test_r2:.4f}")
    print(f"  Training RMSE: ${train_rmse:.4f}K | Testing RMSE: ${test_rmse:.4f}K")
    print(f"  Training MAE: ${train_mae:.4f}K | Testing MAE: ${test_mae:.4f}K")


7.2 MODEL TRAINING RESULTS:
--------------------------------------------------------------------------------

Linear Regression:
  Training R²: 0.8957 | Testing R²: 0.8994
  Training RMSE: $1.6447K | Testing RMSE: $1.7816K
  Training MAE: $1.1985K | Testing MAE: $1.4608K

Ridge Regression (α=1.0):
  Training R²: 0.8957 | Testing R²: 0.8994
  Training RMSE: $1.6447K | Testing RMSE: $1.7816K
  Training MAE: $1.1985K | Testing MAE: $1.4608K

Lasso Regression (α=0.1):
  Training R²: 0.8957 | Testing R²: 0.8996
  Training RMSE: $1.6447K | Testing RMSE: $1.7806K
  Training MAE: $1.1987K | Testing MAE: $1.4598K

Random Forest:
  Training R²: 0.9964 | Testing R²: 0.9815
  Training RMSE: $0.3075K | Testing RMSE: $0.7648K
  Training MAE: $0.2302K | Testing MAE: $0.6143K

Gradient Boosting:
  Training R²: 0.9987 | Testing R²: 0.9831
  Training RMSE: $0.1813K | Testing RMSE: $0.7298K
  Training MAE: $0.1462K | Testing MAE: $0.6187K


# Detailed MODEL EVALUATION 
R² SCORE (Coefficient of Determination):
  - Range: 0 to 1 (higher is better)
  - Interpretation: Proportion of variance in Sales explained by the model
  - R² = 1 means perfect predictions
  - R² = 0 means model is no better than mean prediction
 
RMSE (Root Mean Squared Error):
  - Measures average prediction error in units of target variable
  - Penalizes larger errors more heavily
  - Lower values indicate better fit
  - In this case: measured in thousands of units
 
MAE (Mean Absolute Error):
  - Average absolute difference between predicted and actual values
  - More interpretable than RMSE (in original units)
  - Less sensitive to outliers than RMSE
 
Cross-Validation R² Score:
  - Tests model on multiple data splits to assess generalization
  - 5-Fold CV: splits data into 5 parts, trains 5 times
  - Helps detect overfitting

In [34]:
# Detailed evaluation for best model (Linear Regression for interpretability)
best_model = trained_models['Linear Regression']
y_pred = best_model.predict(X_test)
 
print("\n8.2 LINEAR REGRESSION - DETAILED ANALYSIS:")
print("-" * 80)
 
# Residual analysis
residuals = y_test - y_pred
print(f"Residual Statistics:")
print(f"  Mean: {residuals.mean():.4f} (should be ≈ 0)")
print(f"  Std Dev: {residuals.std():.4f}")
print(f"  Min: {residuals.min():.4f}, Max: {residuals.max():.4f}")
 
# Model coefficients
print(f"\nModel Coefficients (Impact of each channel on Sales):")
print(f"  Intercept: {best_model.intercept_:.4f}")
for feature, coef in zip(X_main.columns, best_model.coef_):
    print(f"  {feature:12} : {coef:.4f} (for every $1K increase in {feature}, sales increase by ${coef:.4f}K)")
 
# Cross-validation
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_model, X_main, y_main, cv=kfold, scoring='r2')
print(f"\n5-Fold Cross-Validation R² Scores: {cv_scores}")
print(f"Mean CV R²: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
 


8.2 LINEAR REGRESSION - DETAILED ANALYSIS:
--------------------------------------------------------------------------------
Residual Statistics:
  Mean: -0.0975 (should be ≈ 0)
  Std Dev: 1.8016
  Min: -3.6035, Max: 2.5876

Model Coefficients (Impact of each channel on Sales):
  Intercept: 2.9791
  TV           : 0.0447 (for every $1K increase in TV, sales increase by $0.0447K)
  Radio        : 0.1892 (for every $1K increase in Radio, sales increase by $0.1892K)
  Newspaper    : 0.0028 (for every $1K increase in Newspaper, sales increase by $0.0028K)

5-Fold Cross-Validation R² Scores: [0.89943802 0.81895573 0.93398674 0.90102145 0.86025006]
Mean CV R²: 0.8827 (+/- 0.0395)


# SALES FORECASTING

In [35]:
# Define scenarios for business decision-making
scenarios = {
    'Conservative (Low Spend)': {'TV': 50, 'Radio': 30, 'Newspaper': 20},
    'Balanced (Medium Spend)': {'TV': 150, 'Radio': 50, 'Newspaper': 50},
    'Aggressive (High Spend)': {'TV': 250, 'Radio': 100, 'Newspaper': 100},
    'TV-Focused': {'TV': 300, 'Radio': 30, 'Newspaper': 30},
    'Digital-Focused': {'TV': 50, 'Radio': 150, 'Newspaper': 50},
}
 
forecast_results = {}
print("\nForecast Results (Using Linear Regression Model):\n")
 
for scenario_name, spend in scenarios.items():
    # Create DataFrame for prediction
    scenario_df = pd.DataFrame([spend])
    predicted_sales = best_model.predict(scenario_df)[0]
    total_spend = sum(spend.values())
    roi = (predicted_sales / total_spend) * 100 if total_spend > 0 else 0
    
    forecast_results[scenario_name] = {
        'spend': spend,
        'predicted_sales': predicted_sales,
        'total_spend': total_spend,
        'roi': roi
    }
    
    print(f"{scenario_name}:")
    print(f"  Advertising Mix: TV=${spend['TV']}K, Radio=${spend['Radio']}K, Newspaper=${spend['Newspaper']}K")
    print(f"  Total Spend: ${total_spend}K")
    print(f"  Predicted Sales: ${predicted_sales:.2f}K units")
    print(f"  ROI (Sales per $ spend): {roi:.2f} units/$K")
    print()


Forecast Results (Using Linear Regression Model):

Conservative (Low Spend):
  Advertising Mix: TV=$50K, Radio=$30K, Newspaper=$20K
  Total Spend: $100K
  Predicted Sales: $10.95K units
  ROI (Sales per $ spend): 10.95 units/$K

Balanced (Medium Spend):
  Advertising Mix: TV=$150K, Radio=$50K, Newspaper=$50K
  Total Spend: $250K
  Predicted Sales: $19.29K units
  ROI (Sales per $ spend): 7.71 units/$K

Aggressive (High Spend):
  Advertising Mix: TV=$250K, Radio=$100K, Newspaper=$100K
  Total Spend: $450K
  Predicted Sales: $33.36K units
  ROI (Sales per $ spend): 7.41 units/$K

TV-Focused:
  Advertising Mix: TV=$300K, Radio=$30K, Newspaper=$30K
  Total Spend: $360K
  Predicted Sales: $22.16K units
  ROI (Sales per $ spend): 6.15 units/$K

Digital-Focused:
  Advertising Mix: TV=$50K, Radio=$150K, Newspaper=$50K
  Total Spend: $250K
  Predicted Sales: $33.73K units
  ROI (Sales per $ spend): 13.49 units/$K

